In [2]:
import os
import numpy as np
import pandas as pd
import tensorflow as tf

# 1. Establish absolute file path configuration
base_dir = r"C:\Users\ACER\tinyml-iot-security"
models_dir = os.path.join(base_dir, "models")
data_path = os.path.join(base_dir, "data", "synthetic_L64.csv")

# Paths for saving models
# Change "baseline_nn_uncompressed.keras" to "tinyml_freeze_replay_model.keras"
keras_model_path = os.path.join(models_dir, "tinyml_freeze_replay_model.keras")
tflite_float_path = os.path.join(models_dir, "freeze_replay_detector_float.tflite")
tflite_quant_path = os.path.join(models_dir, "freeze_replay_detector_quant.tflite")

# 2. Load the trained Day 4 Keras model
print("Loading uncompressed baseline neural network...")
model = tf.keras.models.load_model(keras_model_path)

# 3. Reload a portion of training data for the Day 6 representative dataset calibration
df = pd.read_csv(data_path)
X_raw = df.drop(columns=['label']).values
X_train = X_raw.reshape(-1, 64, 5)  # Restoring shape matching our 5 channels

# =========================================================
# DAY 5: CONVERT TO STANDARD TENSORFLOW LITE (FLOAT)
# =========================================================
print("\n[Day 5] Converting baseline model to standard TFLite flatbuffer...")
converter_float = tf.lite.TFLiteConverter.from_keras_model(model)
tflite_float_model = converter_float.convert()

with open(tflite_float_path, 'wb') as f:
    f.write(tflite_float_model)
print(f" Saved standard TFLite version to: {tflite_float_path}")

# =========================================================
# DAY 6: APPLY FULL INTEGER INT8 QUANTIZATION
# =========================================================
print("\n[Day 6] Initializing Full Integer Quantization process...")
converter_quant = tf.lite.TFLiteConverter.from_keras_model(model)
converter_quant.optimizations = [tf.lite.Optimize.DEFAULT]

# Define calibration generator using your training sensor matrices
def representative_dataset():
    # Loop over 100 sample windows to evaluate dynamic calibration ranges
    for sample in X_train[:100]:
        # Shape structure must match training inputs: (Batch, Steps, Channels)
        yield [sample.reshape(1, 64, 5).astype(np.float32)]

converter_quant.representative_dataset = representative_dataset
converter_quant.target_spec.supported_ops = [tf.lite.OpsSet.TFLITE_BUILTINS_INT8]

# Strictly enforce int8 types for both hardware interfaces
converter_quant.inference_input_type = tf.int8
converter_quant.inference_output_type = tf.int8

tflite_quant_model = converter_quant.convert()

with open(tflite_quant_path, 'wb') as f:
    f.write(tflite_quant_model)
print(f" Saved Full Int8 Quantized version to: {tflite_quant_path}")

# =========================================================
# TRACK COMPRESSION RATIOS
# =========================================================
size_keras = os.path.getsize(keras_model_path) / 1024
size_float = os.path.getsize(tflite_float_path) / 1024
size_quant = os.path.getsize(tflite_quant_path) / 1024

print("\n=== STORAGE SIZE FOOTPRINT COMPARISON ===")
print(f"1. Original Keras CNN Model Size:      {size_keras:.2f} KB")
print(f"2. Standard TFLite (No Quantization):   {size_float:.2f} KB")
print(f"3. Full Integer Quantized TFLite:       {size_quant:.2f} KB")
print(f"Total optimization file size reduction: {((size_keras - size_quant)/size_keras)*100:.1f}% smaller!")


Loading uncompressed baseline neural network...

[Day 5] Converting baseline model to standard TFLite flatbuffer...
INFO:tensorflow:Assets written to: C:\Users\ACER\AppData\Local\Temp\tmpff2xpu8e\assets


INFO:tensorflow:Assets written to: C:\Users\ACER\AppData\Local\Temp\tmpff2xpu8e\assets


Saved artifact at 'C:\Users\ACER\AppData\Local\Temp\tmpff2xpu8e'. The following endpoints are available:

* Endpoint 'serve'
  args_0 (POSITIONAL_ONLY): TensorSpec(shape=(None, 64, 5), dtype=tf.float32, name='input_layer')
Output Type:
  TensorSpec(shape=(None, 3), dtype=tf.float32, name=None)
Captures:
  2820365265296: TensorSpec(shape=(), dtype=tf.resource, name=None)
  2820365267024: TensorSpec(shape=(), dtype=tf.resource, name=None)
  2820365267216: TensorSpec(shape=(), dtype=tf.resource, name=None)
  2820365264912: TensorSpec(shape=(), dtype=tf.resource, name=None)
  2820365268560: TensorSpec(shape=(), dtype=tf.resource, name=None)
  2820365268176: TensorSpec(shape=(), dtype=tf.resource, name=None)
  2820365269904: TensorSpec(shape=(), dtype=tf.resource, name=None)
  2820365270864: TensorSpec(shape=(), dtype=tf.resource, name=None)
 Saved standard TFLite version to: C:\Users\ACER\tinyml-iot-security\models\freeze_replay_detector_float.tflite

[Day 6] Initializing Full Integer Quan

INFO:tensorflow:Assets written to: C:\Users\ACER\AppData\Local\Temp\tmpgw7m6xs4\assets


Saved artifact at 'C:\Users\ACER\AppData\Local\Temp\tmpgw7m6xs4'. The following endpoints are available:

* Endpoint 'serve'
  args_0 (POSITIONAL_ONLY): TensorSpec(shape=(None, 64, 5), dtype=tf.float32, name='input_layer')
Output Type:
  TensorSpec(shape=(None, 3), dtype=tf.float32, name=None)
Captures:
  2820365265296: TensorSpec(shape=(), dtype=tf.resource, name=None)
  2820365267024: TensorSpec(shape=(), dtype=tf.resource, name=None)
  2820365267216: TensorSpec(shape=(), dtype=tf.resource, name=None)
  2820365264912: TensorSpec(shape=(), dtype=tf.resource, name=None)
  2820365268560: TensorSpec(shape=(), dtype=tf.resource, name=None)
  2820365268176: TensorSpec(shape=(), dtype=tf.resource, name=None)
  2820365269904: TensorSpec(shape=(), dtype=tf.resource, name=None)
  2820365270864: TensorSpec(shape=(), dtype=tf.resource, name=None)


c:\Users\ACER\tinyml-iot-security\venv\Lib\site-packages\tensorflow\lite\python\convert.py:846: UserWarning: Statistics for quantized inputs were expected, but not specified; continuing anyway.
  warnings.warn(


 Saved Full Int8 Quantized version to: C:\Users\ACER\tinyml-iot-security\models\freeze_replay_detector_quant.tflite

=== STORAGE SIZE FOOTPRINT COMPARISON ===
1. Original Keras CNN Model Size:      229.69 KB
2. Standard TFLite (No Quantization):   70.29 KB
3. Full Integer Quantized TFLite:       25.52 KB
Total optimization file size reduction: 88.9% smaller!


In [5]:
import os
import time
import numpy as np
import pandas as pd
import tensorflow as tf
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score

# 1. Re-establish paths and reload data
base_dir = r"C:\Users\ACER\tinyml-iot-security"
models_dir = os.path.join(base_dir, "models")
data_path = os.path.join(base_dir, "data", "synthetic_L64.csv")

df = pd.read_csv(data_path)
X_raw = df.drop(columns=['label']).values
y_raw = df['label'].values

# Recreate matrix structures
num_samples = X_raw.shape[0]
X_reshaped = X_raw.reshape(num_samples, 64, 5)

X_train, X_temp, y_train, y_temp = train_test_split(X_reshaped, y_raw, test_size=0.30, random_state=42, stratify=y_raw)
X_val, X_test, y_val, y_test = train_test_split(X_temp, y_temp, test_size=0.50, random_state=42, stratify=y_temp)

scaler = StandardScaler()
X_train_flat = X_train.reshape(-1, 5)
scaler.fit(X_train_flat)

X_test_flat = X_test.reshape(-1, 5)
X_test_scaled = scaler.transform(X_test_flat).reshape(X_test.shape)

# Define TFLite paths
tflite_float_path = os.path.join(models_dir, "freeze_replay_detector_float.tflite")
tflite_quant_path = os.path.join(models_dir, "freeze_replay_detector_quant.tflite")
keras_model_path = os.path.join(models_dir, "tinyml_freeze_replay_model.keras")

# Ensure sizes are calculated for display
size_keras = os.path.getsize(keras_model_path) / 1024
size_float = os.path.getsize(tflite_float_path) / 1024
size_quant = os.path.getsize(tflite_quant_path) / 1024

# Explicitly cast to float32
X_test_float32 = X_test_scaled.astype(np.float32)

# =========================================================
# BENCHMARK #1: STANDARD FLOAT TFLITE INTERPRETER
# =========================================================
interpreter_float = tf.lite.Interpreter(model_path=tflite_float_path)
interpreter_float.allocate_tensors()

# FIX: Access the first element [0] of the returned list
input_details_f = interpreter_float.get_input_details()[0]
output_details_f = interpreter_float.get_output_details()[0]

y_pred_float = []
start_time = time.time()

for sample in X_test_float32:
    interpreter_float.set_tensor(input_details_f['index'], sample.reshape(1, 64, 5))
    interpreter_float.invoke()
    output_data = interpreter_float.get_tensor(output_details_f['index'])
    y_pred_float.append(np.argmax(output_data))

float_total_time = (time.time() - start_time) * 1000
float_avg_time = float_total_time / len(X_test_float32)
float_acc = accuracy_score(y_test, y_pred_float) * 100

# =========================================================
# BENCHMARK #2: FULL INTEGER INT8 TFLITE INTERPRETER
# =========================================================
interpreter_quant = tf.lite.Interpreter(model_path=tflite_quant_path)
interpreter_quant.allocate_tensors()

# FIX: Access the first element [0] of the returned list
input_details_q = interpreter_quant.get_input_details()[0]
output_details_q = interpreter_quant.get_output_details()[0]

input_scale, input_zero_point = input_details_q['quantization']

y_pred_quant = []
start_time = time.time()

for sample in X_test_float32:
    if input_scale != 0.0:
        sample_quant = (sample / input_scale) + input_zero_point
    else:
        sample_quant = sample
    sample_quant = np.round(sample_quant).astype(np.int8)
    
    interpreter_quant.set_tensor(input_details_q['index'], sample_quant.reshape(1, 64, 5))
    interpreter_quant.invoke()
    
    output_data = interpreter_quant.get_tensor(output_details_q['index'])
    y_pred_quant.append(np.argmax(output_data))

quant_total_time = (time.time() - start_time) * 1000
quant_avg_time = quant_total_time / len(X_test_float32)
quant_acc = accuracy_score(y_test, y_pred_quant) * 100

# =========================================================
# PRINT FINAL COMPARISON TABLE 
# =========================================================
print("\n" + "="*65)
print(f"{'MODEL VARIANT':<22} | {'SIZE (KB)':<10} | {'ACCURACY':<10} | {'LATENCY/SAMPLE':<12}")
print("="*65)
print(f"{'Random Forest Baseline':<22} | {'~150 KB':<10} | {'99.50%':<10} | {'~0.05 ms':<12}")
print(f"{'Full Keras 1D-CNN':<22} | {f'{size_keras:.1f} KB':<10} | {'100.00%':<10} | {'~0.22 ms':<12}")
print(f"{'TFLite (Standard Float)':<22} | {f'{size_float:.1f} KB':<10} | {f'{float_acc:.2f}%':<10} | {f'{float_avg_time:.3f} ms':<12}")
print(f"{'TFLite (Quantized Int8)':<22} | {f'{size_quant:.1f} KB':<10} | {f'{quant_acc:.2f}%':<10} | {f'{quant_avg_time:.3f} ms':<12}")
print("="*65)



MODEL VARIANT          | SIZE (KB)  | ACCURACY   | LATENCY/SAMPLE
Random Forest Baseline | ~150 KB    | 99.50%     | ~0.05 ms    
Full Keras 1D-CNN      | 229.7 KB   | 100.00%    | ~0.22 ms    
TFLite (Standard Float) | 70.3 KB    | 89.00%     | 0.049 ms    
TFLite (Quantized Int8) | 25.5 KB    | 20.00%     | 0.023 ms    


c:\Users\ACER\tinyml-iot-security\venv\Lib\site-packages\tensorflow\lite\python\interpreter.py:457: UserWarning:     Warning: tf.lite.Interpreter is deprecated and is scheduled for deletion in
    TF 2.20. Please use the LiteRT interpreter from the ai_edge_litert package.
    See the [migration guide](https://ai.google.dev/edge/litert/migration)
    for details.
    
  warnings.warn(_INTERPRETER_DELETION_WARNING)
c:\Users\ACER\tinyml-iot-security\venv\Lib\site-packages\tensorflow\lite\python\interpreter.py:457: UserWarning:     Warning: tf.lite.Interpreter is deprecated and is scheduled for deletion in
    TF 2.20. Please use the LiteRT interpreter from the ai_edge_litert package.
    See the [migration guide](https://ai.google.dev/edge/litert/migration)
    for details.
    
  warnings.warn(_INTERPRETER_DELETION_WARNING)
